In [1]:
import pandas as pd
import numpy as np

In [2]:
from ISLP import load_data
from ISLP.models import (ModelSpec as MS, summarize)

In [3]:
import statsmodels.api as sm

# Credit Card Default Dataset

In [4]:
default = load_data("Default")
default

,default,student,balance,income
0,No,No,729.526495,44361.625074
1,No,Yes,817.180407,12106.134700
2,No,No,1073.549164,31767.138947
3,No,No,529.250605,35704.493935
4,No,No,785.655883,38463.495879
...,...,...,...,...
9995,No,No,711.555020,52992.378914
9996,No,No,757.962918,19660.721768
9997,No,No,845.411989,58636.156984
9998,No,No,1569.009053,36669.112365


# (a)

In [5]:
design = MS(["balance", "income"])
X = design.fit_transform(default)
y = default["default"] == "Yes"
model = sm.GLM(y, X, family=sm.families.Binomial())
results = model.fit()
summarize(results)

,coef,std err,z,P>|z|
intercept,-11.540500,0.435000,-26.544,0.0
balance,0.005600,0.000000,24.835,0.0
income,0.000021,0.000005,4.174,0.0


In [11]:
results.params

intercept   -11.540468
balance       0.005647
income        0.000021
dtype: float64

# (b)

In [14]:
def boot_fn(data, idx):
    design = MS(["balance", "income"])
    X = design.fit_transform(data.iloc[idx])
    y = data.iloc[idx]["default"] == "Yes"
    model = sm.GLM(y, X, family=sm.families.Binomial())
    results = model.fit()
    return results.params

# (c)

In [25]:
rng = np.random.default_rng(1)
B = 1000 # no. of bootstrap sample
coefs = []
for _ in range(B):
    idx = rng.choice(default.shape[0], default.shape[0], replace=True)
    params = boot_fn(default, idx)
    coefs.append(params)
coefs

[intercept   -11.691159
 balance       0.005789
 income        0.000018
 dtype: float64,
 intercept   -11.913753
 balance       0.005711
 income        0.000027
 dtype: float64,
 intercept   -11.018952
 balance       0.005575
 income        0.000009
 dtype: float64,
 intercept   -11.537299
 balance       0.005588
 income        0.000027
 dtype: float64,
 intercept   -11.176961
 balance       0.005462
 income        0.000017
 dtype: float64,
 intercept   -11.817188
 balance       0.005857
 income        0.000020
 dtype: float64,
 intercept   -12.161960
 balance       0.005816
 income        0.000032
 dtype: float64,
 intercept   -12.421179
 balance       0.006096
 income        0.000025
 dtype: float64,
 intercept   -11.163702
 balance       0.005376
 income        0.000022
 dtype: float64,
 intercept   -11.345740
 balance       0.005603
 income        0.000017
 dtype: float64,
 intercept   -12.705349
 balance       0.006154
 income        0.000032
 dtype: float64,
 intercept   -11.4336

In [29]:
coefs = np.array(coefs)
np.std(coefs, axis=0)

array([4.38104183e-01, 2.25799772e-04, 4.87116546e-06])

# (d)

The SE calculated using bootstrap is very close to the actual SE given by sm.GLE method.